# Tutorial 0: Onboarding Roadmap

## What is Bayesian metamodeling?

Imagine you have four separate models of early T-cell receptor signaling — one for membrane geometry, one for CD45 segregation, one for Lck activity, one for TCR phosphorylation. Each is independently parameterized and reasonably good in its domain, but they don't talk to each other. **Bayesian metamodeling is how you make them agree, propagate uncertainty across them, and produce a *joint posterior* over what all four are saying together.** This curriculum walks you from a 9-point toy sweep to that point in 9 tutorials.

The framework is **spec-driven** (one typed JSON contract per model), **CLI-first** (`bayesmm validate / plan / run / surrogate / meta`), and treats every run as a reproducible experiment with full provenance. The destination of this curriculum is `projects/tcr_signaling/` — the read-only submodule reproducing Neve-Oz, Sherman & Raveh (Frontiers in Immunology, 2024). You'll see those four models again in Tutorial 9.


## Environment first (simple rule)

Use a Jupyter kernel that already belongs to the conda (or venv) environment you
want. That kernel environment is what persists across all notebook cells.

You don't need to memorize environment names — the preflight cell below runs
**`bayesmm doctor`**, which reports your actual OS, Python, environment, and which
optional backends (PyMC / SBI) are installed. If something is missing, run
**`bayesmm setup`** for platform-correct install commands.

No installation commands run automatically in this notebook.


## Vocabulary you'll see across these tutorials

Definitions are intentionally one-sentence each. Later tutorials reference these terms; if you're confused mid-tutorial, come back here.

- **Spec**: a typed JSON contract describing one model — its identity, IO schema, runner, adapter, DOE plan, and storage. Validated by `bayesmm validate`.
- **DOE** (*design of experiments*): the set of input points at which you'll run the model. Two strategies in this curriculum: `grid` (cartesian product) and `sobol` (deterministic space-filling).
- **Sweep**: one execution of a model across all DOE points. Produces a centralized `sweep_rows.csv` with one row per point.
- **Adapter**: the small piece of code that turns a DOE point into a process invocation (e.g. `python_cli_adapter_v1`, `biomodels_sbml_adapter_v1`).
- **Surrogate**: a fast probabilistic model fit to a sweep's outputs that lets you predict at arbitrary inputs without re-running the simulator. Two backends here: PyMC GP and SBI NPE.
- **Coupling**: a constraint linking variables across models. **Deterministic** coupling propagates a value (`z = α·C + β`); **probabilistic** coupling propagates a distribution (e.g. `y ~ Normal(C, σ)`). The two have qualitatively different effects on the joint.
- **Metamodel**: the composed object that wires multiple surrogates together via couplings and produces a joint posterior when sampled.
- **Joint posterior**: the multi-variable distribution the metamodel produces when sampled — what your coupled models believe *together*, including the correlations the couplings induce.
- **Posterior predictive**: the surrogate's belief about the model output at a new input, expressed as a distribution (mean + width), not a point estimate. The width is the lesson.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent

os.chdir(project_root)
src_path = project_root / "src"
if src_path.is_dir() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

env = os.environ.copy()
src_str = str(src_path)
# os.pathsep is ":" on POSIX and ";" on Windows — never hardcode the separator.
env["PYTHONPATH"] = (
    src_str
    if not env.get("PYTHONPATH")
    else os.pathsep.join([src_str, env["PYTHONPATH"]])
)

print(f"Project root: {project_root}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

# CLI preflight — confirms the package is importable in this kernel.
result = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"],
    cwd=project_root,
    env=env,
    text=True,
    capture_output=True,
)
if result.returncode != 0:
    raise RuntimeError(
        "CLI preflight failed. Make sure this notebook runs in a prepared environment.\n"
        f"stderr:\n{result.stderr}"
    )
print(result.stdout.strip())

# Environment diagnostic — `bayesmm doctor` reports OS, Python, conda/venv, and
# which optional backends (pymc / sbi / torch) are available. This is the fastest
# way to confirm your kernel is ready for the tutorials below; if a backend is
# missing, run `bayesmm setup` for platform-correct install commands.
doctor = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "doctor"],
    cwd=project_root,
    env=env,
    text=True,
    capture_output=True,
)
print()
print(doctor.stdout.strip() or doctor.stderr.strip())


Project root: /Users/barakraveh/Git/metamodeler_codex_scaffold_docs
Python executable: /Users/barakraveh/miniconda3/envs/py312_bayesmm_sbi/bin/python3.12
Python version: 3.12.12


bayesian-metamodeling 0.1.0



Bayesian Metamodeling environment diagnostic
OS:        Darwin 25.4.0 (arm64)
Python:    CPython 3.12.12 (/Users/barakraveh/miniconda3/envs/py312_bayesmm_sbi/bin/python3.12)
Env:       conda (py312_bayesmm_sbi) -> /Users/barakraveh/miniconda3/envs/py312_bayesmm_sbi
Tools:     pip=26.0.1, conda=conda 26.1.1
Shell:     zsh (interactive=False)
Repo root: /Users/barakraveh/Git/metamodeler_codex_scaffold_docs (writable=True)

Optional backends
----------------------------------------
  [OK]   arviz    0.23.4
  [OK]   pymc     5.28.1
  [OK]   sbi      0.25.0
  [OK]   torch    2.5.1


## Optional commands (manual, uncomment if needed)

Use these in a terminal or uncomment in a notebook cell if you explicitly want them.
Adjust environment names to your setup.


In [2]:
## `bayesmm setup` generates the correct install commands for *your* platform
## (conda vs pip, Windows vs macOS/Linux quoting). Run it instead of guessing:
#   python -m bayesian_metamodeling.cli.main setup
#
## Or install manually in a terminal (adjust to your environment):
## PyMC backend  — learning/coupling with PyMC:
#   conda install -c conda-forge pymc arviz       # or: pip install 'bayesian-metamodeling[pymc]'
## SBI backend   — learning/coupling with SBI:
#   conda install -c conda-forge pytorch sbi      # or: pip install 'bayesian-metamodeling[sbi]'
## BioModels (Tutorial 2) — SBML model execution:
#   pip install libroadrunner tellurium

## Package status
Report what packages are available in current environment

In [3]:
import importlib.util
import importlib.metadata as md

packages = ["pymc", "arviz", "torch", "sbi", "tellurium"]
status = {}
for pkg in packages:
    if importlib.util.find_spec(pkg) is None:
        status[pkg] = "not_installed"
    else:
        try:
            status[pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            status[pkg] = "installed"

for pkg, ver in status.items():
    # print right aligned package name and version
    print(f"{pkg:>20}: {ver}")


# Print libroadrunner version using roadrunner API if available
try:
    import roadrunner
    print(f"{'libroadrunner (API)':>20}: {roadrunner.__version__}")
except ImportError:
    print(f"{'libroadrunner (API)':>20}: not_installed")

                pymc: 5.28.1
               arviz: 0.23.4
               torch: 2.5.1
                 sbi: 0.25.0
           tellurium: not_installed
 libroadrunner (API): not_installed


## Tutorial map

Each tutorial answers one question and leaves you with one new capability. Don't worry about the time estimates — they're rough.

| # | What you build | What you'll be able to do |
|---|---|---|
| 1 | Run a 9-point toy sweep, plot two heatmaps from one centralized CSV | Drive the `validate → plan → run` CLI loop end-to-end |
| 2 | Run a real published BioModels SBML model with a dense `k_on` sweep | Recognize the spec contract is the same shape for toy and real models |
| 3 | Break a spec on purpose, then read and fix the validator output | Read a Pydantic validation error and fix the spec without trial-and-error |
| 4 | Compare `grid` vs `sobol` DOE strategies on the same toy | Pick the right DOE strategy for your model's dimensionality |
| 5 | Fit a PyMC GP surrogate, evaluate it on new inputs with uncertainty | Read a posterior-predictive width and decide whether to trust a prediction |
| 6 | Fit an SBI NPE surrogate on the same data, compare to PyMC | Tell when the two backends agree (and what disagreement means) |
| 7 | Couple two surrogates with `equality_soft`, sample the joint | See what "agreement between models" looks like as a scatter cloud |
| 8 | Chain three surrogates, observe uncertainty propagation along the chain | Budget noise across a coupled cascade |
| 9 | Compose your own DOE → surrogate → metamodel, write a 3-sentence report | Drive the framework end-to-end without copy-paste — the capstone |

After T9, the natural next step is `projects/tcr_signaling/` — the same workflow at full scale on four real biological models.


## How the pieces fit together

This is the data flow each tutorial slots into. Tutorials 1-2 cover the left side (spec → run → CSV). Tutorials 3-4 are about specs and DOE planning. Tutorials 5-6 are the middle (CSV → surrogate). Tutorials 7-8 are the right side (surrogates → coupling → joint metamodel). Tutorial 9 composes the whole thing end-to-end.

```
┌────────┐   ┌──────────┐   ┌────────────┐   ┌──────────────────┐   ┌─────────────┐   ┌──────────────┐
│  Spec  │ → │ DOE plan │ → │   Sweep    │ → │ sweep_rows.csv   │ → │  Surrogate  │ → │   Metamodel  │
│ (JSON) │   │  (grid/  │   │ (executes  │   │ (centralized,    │   │  artifact   │   │  + couplings │
│        │   │   sobol) │   │  N points) │   │  one row/point)  │   │ (.artifact) │   │              │
└────────┘   └──────────┘   └────────────┘   └──────────────────┘   └─────────────┘   └──────────────┘
   T3            T4              T1, T2                                  T5, T6               T7, T8
                                                                                                │
                                                                                                ▼
                                                                                      ┌──────────────────┐
                                                                                      │  Joint posterior │
                                                                                      │ (samples_dataset │
                                                                                      │     .json)       │
                                                                                      └──────────────────┘
                                                                                              T9
```

Three storage artifacts you'll see across the tutorials: **`sweep_rows.csv`** (one per sweep, the canonical handoff to surrogates), **`.artifact.json`** (one per fitted surrogate, contains the trained model + provenance), and **`samples_dataset.json`** (one per metamodel sampling run, contains posterior draws).


## Tips for new lab members

- **Keep a short run log**: commands, run IDs, and a one-line interpretation per major step. The framework's run registry tracks IDs, but your interpretation notes are what make a sweep reproducible six months later.
- **If a dependency is missing, continue with fallback paths.** If `bayesmm doctor` reports SBI missing and you don't plan to do T6, that's fine — skip T6 for now. If `tellurium`/`libroadrunner` are missing, T2 will skip its `bayesmm run` cell with a clear banner — you can still inspect the spec and plan output. Come back when you have time to install.
- **`MM_BIOMODELS_OFFLINE=1` for sandboxed runs.** If you're on a machine without internet (or you don't want to download a 1.4 MB SBML on the spot), set this env var and T2's adapter will refuse to download — it'll either use a cached SBML you already have or print a `curl` recipe so you can pre-populate the cache from a different machine.
- **Inspect one artifact file after each major command.** `cat tmp/run_registry.json | head` after a sweep, `ls tmp/surrogate_artifacts/*/artifact.json` after a fit. Looking at the artifact once teaches you more than reading the docs about it.
- **Time sinks to know about**: T2's first run downloads ~1.4 MB from BioModels (5-30 s). T5 and T6's surrogate fits take 10-60 s each. T8's joint sampling takes 30-90 s on a CPU. Everything else is sub-second.
- **The destination is `projects/tcr_signaling/`.** When this curriculum starts feeling like toy work, that's the signal you're ready to read its README.
